# Exploración: OECE / SEACE - Buscador de Proveedores Adjudicados (Pentaho BI)

- **Entidad oficial:** Organismo Especializado para las Contrataciones Públicas Eficientes (OECE) / Sistema Electrónico de Contrataciones del Estado (SEACE).
- **Descripción y propósito:** Plataforma analítica oficial que expone los antecedentes de proveedores del Estado peruano. Permite descubrir en segundos todas las personas jurídicas (empresas) o registros personales vinculados a un ciudadano, auditar su composición societaria (accionistas, representantes y órganos de administración) y reconstruir su historial de buenas pro y órdenes de compra/servicio desde el año 2008, siendo una fuente crítica para detectar redes de empresas vinculadas o colusión en licitaciones.
- **Enlace del portal:** `https://bi.seace.gob.pe/pentaho/api/repos/:public:ANTECEDENTES_PROVEEDORES:ANTECEDENTES_PROVEEDORES.wcdf/generatedContent?userid=public&password=key`
- **Endpoint directo (API CDA):** `https://bi.seace.gob.pe/pentaho/plugin/cda/api/doQuery?userid=public&password=key`
- **Criterios de búsqueda soportados:**
  - **Por RUC (11 dígitos):** Consulta directa sobre las tablas de detalle del proveedor.
  - **Por DNI (8 dígitos):** Búsqueda relacional en cascada (estructura de árbol) que identifica primero todos los RUC donde la persona natural tiene presencia (como titular, accionista, representante legal o miembro de directorio) y extrae el detalle completo de cada entidad vinculada.
- **Información que se obtiene:**
  - **Presencia en el RNP (Tabla maestra por DNI/RUC):** RUC del proveedor vinculado, Razón Social y condición general (`VIGENTE`, etc.).
  - **Vigencias en el RNP:** Capítulo (`Proveedor de Bienes`, `Proveedor de Servicios`), estado del capítulo, fecha de inicio, fecha de fin (`INDETERMINADO` o rango) y departamento registrado.
  - **Composición de Accionistas y Representantes Legales:** Tipo de vínculo (`ACCIONISTA`, `REPRESENTANTE`, `ORG. ADMINISTRACION`), número de documento o RUC, nombre completo o razón social y porcentaje de participación accionaria (`% ACCIONES`).
  - **Historial de Adjudicaciones (desde 2008):** Año/periodo, modalidad de contratación, nombre de la entidad pública contratante, monto total adjudicado en soles (`S/`) y cantidad de procesos u órdenes.

## Estrategia de Extracción Técnica

- **Arquitectura del portal:** Dashboard analítico construido sobre **Pentaho Business Analytics (Ctools / CDF)** que consume datos en segundo plano mediante el conector **CDA (`Community Data Access`)** sobre el archivo de definición `/public/ANTECEDENTES_PROVEEDORES/ANTECEDENTES_PROVEEDORES.cda`.
- **Consumo directo de API sin navegador (`httpx`):**
  1. En lugar de renderizar la interfaz gráfica ni simular clics en las tablas con `Playwright`, se invoca directamente el endpoint `POST /pentaho/plugin/cda/api/doQuery?userid=public&password=key` enviando payloads `application/x-www-form-urlencoded`.
  2. El motor CDA responde estructuras JSON matriciales compuestas por `metadata` (definición de columnas) y `resultset` (filas de datos), las cuales se transforman dinámicamente en diccionarios clave-valor mediante mapeo posicional (`zip`).
- **Identificadores de consulta (`dataAccessId`) descubiertos:**
  - `QUERY_R_F0` (con parámetro `paramPARAM_DNIRUC0`): Consulta maestra que retorna la tabla *"Presencia en el Registro Nacional de Proveedores"*.
  - `QUERY_R_F3` (con parámetro `paramPARAM_RUC`): Retorna las vigencias por capítulo en el RNP.
  - `QUERY_R_F2_new` (con parámetro `paramPARAM_RUC`): Retorna la composición societaria (accionistas, administración y representantes).
  - `QUERY_R_F5_D_DASH_PROV_RESUMEN` (con parámetro `paramPARAM_RUC`): Retorna el consolidado histórico de adjudicaciones y órdenes por entidad y año.
- **Orquestación asíncrona en árbol (`asyncio.gather`):**
  - En consultas por **RUC**, se lanzan en paralelo las 3 consultas de detalle (`QUERY_R_F3`, `QUERY_R_F2_new` y `QUERY_R_F5_D_DASH_PROV_RESUMEN`), obteniendo toda la ficha en `~300 ms`.
  - En consultas por **DNI**, se ejecuta primero `QUERY_R_F0` (`paramPARAM_DNIRUC0`) para descubrir los $N$ proveedores asociados al documento e inmediatamente se dispara un `asyncio.gather` anidado que extrae en paralelo las 3 sub-tablas de cada empresa vinculada, construyendo el árbol societario y contractual completo en menos de 1 segundo.

In [4]:
import asyncio
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any, List

# Configuración exclusiva para consulta por RUC (11 dígitos)
RUC_PRUEBA = "20607524476"

URL_CDA_AUTH = "https://bi.seace.gob.pe/pentaho/plugin/cda/api/doQuery?userid=public&password=key"
HEADERS_SEACE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://bi.seace.gob.pe",
    "Referer": "https://bi.seace.gob.pe/pentaho/api/repos/:public:ANTECEDENTES_PROVEEDORES:ANTECEDENTES_PROVEEDORES.wcdf/generatedContent?userid=public&password=key"
}

QUERIES = {
    "socios": "QUERY_R_F2_new",
    "vigencias": "QUERY_R_F3",
    "adjudicaciones": "QUERY_R_F5_D_DASH_PROV_RESUMEN"
}

def matricial_a_dict(data_json: dict) -> List[Dict[str, Any]]:
    if not data_json or "resultset" not in data_json:
        return []
    columnas = [col["colName"].strip().upper() for col in data_json.get("metadata", [])]
    return [dict(zip(columnas, fila)) for fila in data_json.get("resultset", [])]

async def pedir_cda(client: httpx.AsyncClient, ruc: str, query_id: str) -> Dict[str, Any]:
    form_data = {
        "path": "/public/ANTECEDENTES_PROVEEDORES/ANTECEDENTES_PROVEEDORES.cda",
        "dataAccessId": query_id,
        "paramPARAM_RUC": ruc.strip(),
        "outputIndexId": "1",
        "pageSize": "0",
        "pageStart": "0",
        "paramsearchBox": ""
    }
    try:
        resp = await client.post(URL_CDA_AUTH, data=form_data, timeout=25.0)
        return {"exito": resp.status_code == 200, "raw": resp.json() if resp.status_code == 200 else None}
    except Exception as exc:
        return {"exito": False, "raw": None, "error": str(exc)}

async def consultar_seace_pentaho(ruc: str) -> Dict[str, Any]:
    ruc_limpio = ruc.strip()
    if len(ruc_limpio) != 11 or not ruc_limpio.isdigit():
        return {"error": "La consulta en SEACE Pentaho CDA requiere exclusivamente un RUC numérico de 11 dígitos."}

    print(f"[1/2] Consultando SEACE Pentaho CDA para RUC: {ruc_limpio}...")

    inicio = datetime.now(timezone.utc)
    async with httpx.AsyncClient(headers=HEADERS_SEACE) as client:
        res_soc, res_vig, res_adj = await asyncio.gather(
            pedir_cda(client, ruc_limpio, QUERIES["socios"]),
            pedir_cda(client, ruc_limpio, QUERIES["vigencias"]),
            pedir_cda(client, ruc_limpio, QUERIES["adjudicaciones"]),
            return_exceptions=True
        )
    duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)

    socios = [
        {
            "relationship_type": r.get("TIPO"),
            "document_number": str(r.get("NRO_DOC", "")).strip(),
            "full_name": r.get("NOMBRE O RAZON SOCIAL"),
            "shares_percentage": float(r.get("ACCIONES", 0.0) or 0.0)
        }
        for r in matricial_a_dict(res_soc.get("raw") if isinstance(res_soc, dict) else None)
    ]

    vigencias = [
        {
            "chapter": r.get("CAPITULORNP"),
            "status": r.get("ESTADO2"),
            "start_date": r.get("FINICIO"),
            "end_date": r.get("FFIN"),
            "department": r.get("DEPARTAMENTOOEE")
        }
        for r in matricial_a_dict(res_vig.get("raw") if isinstance(res_vig, dict) else None)
    ]

    adjudicaciones = [
        {
            "period": str(r.get("PERIODO")),
            "modality": r.get("MODALIDAD"),
            "entity": r.get("ENTIDAD"),
            "amount": float(r.get("MONTO", 0.0) or 0.0),
            "processes_count": int(r.get("CONVOCATORIAS", 0) or 0)
        }
        for r in matricial_a_dict(res_adj.get("raw") if isinstance(res_adj, dict) else None)
    ]

    exito_datos = bool(socios or vigencias or adjudicaciones)
    print(f"[2/2] Extracción finalizada en {duracion} ms | Vigencias: {len(vigencias)} | Socios: {len(socios)} | Adjudicaciones: {len(adjudicaciones)}")

    return {
        "audit": {
            "source": "SEACE_PENTAHO_CDA",
            "tax_id": ruc_limpio,
            "duration_ms": duracion,
            "status": "SUCCESS" if exito_datos else "NO_DATA"
        },
        "data": {
            "ruc": ruc_limpio,
            "rnp_validities": vigencias,
            "shareholders": socios,
            "awards_history": adjudicaciones
        }
    }

# Ejecución en el notebook
resultado_seace = await consultar_seace_pentaho(RUC_PRUEBA)

print("\n=================== 1. VIGENCIAS RNP ===================")
for v in resultado_seace.get("data", {}).get("rnp_validities", []):
    print(f"- [{v['status']}] {v['chapter']} | Del {v['start_date']} al {v['end_date']} ({v['department']})")

print("\n============= 2. COMPOSICIÓN SOCIETARIA =============")
for s in resultado_seace.get("data", {}).get("shareholders", []):
    print(f"- [{s['relationship_type']}] {s['full_name']} | Doc: {s['document_number']} | {s['shares_percentage']}%")

print("\n============ 3. HISTORIAL ADJUDICACIONES ============")
for a in resultado_seace.get("data", {}).get("awards_history", []):
    print(f"- Año {a['period']} | {a['entity']} | S/ {a['amount']:,.2f} ({a['processes_count']} convocatorias)")

[1/2] Consultando SEACE Pentaho CDA para RUC: 20607524476...
[2/2] Extracción finalizada en 287 ms | Vigencias: 2 | Socios: 3 | Adjudicaciones: 38

=================== 1. VIGENCIAS RNP ===================
- [VIGENTE] Proveedor de Servicios | Del 25/03/2021 al INDETERMINADO (CUSCO)
- [VIGENTE] Proveedor de Bienes | Del 25/03/2021 al INDETERMINADO (CUSCO)

============= 2. COMPOSICIÓN SOCIETARIA =============
- [ACCIONISTA] RUIZ AUCCAPIÑA RONALD | Doc: 44900952 | 100.0%
- [ORG. ADMINISTRACION] RUIZ AUCCAPIÑA RONALD | Doc: 44900952 | 0.0%
- [REPRESENTANTE] RUIZ AUCCAPIÑA RONALD | Doc: 44900952 | 0.0%

============ 3. HISTORIAL ADJUDICACIONES ============
- Año 2024 | MUNICIPALIDAD PROVINCIAL DE CALCA | S/ 2,484.00 (1 convocatorias)
- Año 2023 | MUNICIPALIDAD PROVINCIAL DE CALCA | S/ 322,592.71 (10 convocatorias)
- Año 2023 | MUNICIPALIDAD DISTRITAL DE OCONGATE | S/ 238,632.24 (6 convocatorias)
- Año 2023 | MUNICIPALIDAD DISTRITAL DE OCONGATE | S/ 236,006.40 (1 convocatorias)
- Año 2023 | 

In [6]:
import asyncio
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any, List
from playwright.async_api import async_playwright

DNI_PRUEBA = "44900952"

URL_DASHBOARD = "https://bi.seace.gob.pe/pentaho/api/repos/:public:ANTECEDENTES_PROVEEDORES:ANTECEDENTES_PROVEEDORES.wcdf/generatedContent?userid=public&password=key"
URL_CDA_DOQUERY = "https://bi.seace.gob.pe/pentaho/plugin/cda/api/doQuery?userid=public&password=key"
URL_CDA_LIST = "https://bi.seace.gob.pe/pentaho/plugin/cda/api/listQueries?path=/public/ANTECEDENTES_PROVEEDORES/ANTECEDENTES_PROVEEDORES.cda&userid=public&password=key"

HEADERS_SEACE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://bi.seace.gob.pe",
    "Referer": URL_DASHBOARD
}

QUERIES_DETALLE = {
    "socios": "QUERY_R_F2_new",
    "vigencias": "QUERY_R_F3",
    "adjudicaciones": "QUERY_R_F5_D_DASH_PROV_RESUMEN"
}

def matricial_a_dict(data_json: dict) -> List[Dict[str, Any]]:
    if not data_json or "resultset" not in data_json:
        return []
    columnas = [col["colName"].strip().upper() for col in data_json.get("metadata", [])]
    return [dict(zip(columnas, fila)) for fila in data_json.get("resultset", [])]

async def pedir_cda(client: httpx.AsyncClient, query_id: str, params_extra: Dict[str, str]) -> Dict[str, Any]:
    form_data = {
        "path": "/public/ANTECEDENTES_PROVEEDORES/ANTECEDENTES_PROVEEDORES.cda",
        "dataAccessId": query_id,
        "outputIndexId": "1",
        "pageSize": "0",
        "pageStart": "0",
        "paramsearchBox": ""
    }
    form_data.update(params_extra)
    try:
        resp = await client.post(URL_CDA_DOQUERY, data=form_data, timeout=25.0)
        return {"exito": resp.status_code == 200, "raw": resp.json() if resp.status_code == 200 else None}
    except Exception as exc:
        return {"exito": False, "raw": None, "error": str(exc)}

async def descubrir_busqueda_dni_pentaho(dni: str) -> List[Dict[str, str]]:
    """
    Ingresa el DNI en el buscador real de Pentaho e intercepta la respuesta JSON de la primera tabla
    ('Presencia en el Registro Nacional de Proveedores') junto con sus parámetros exactos.
    """
    proveedores = []
    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(headless=True, channel="chrome")
        except Exception:
            browser = await p.chromium.launch(headless=True, channel="msedge")

        page = await browser.new_page()
        respuestas_cda = []

        async def interceptar_doquery(response):
            if "doQuery" in response.url and response.status == 200:
                try:
                    post_data = response.request.post_data or ""
                    json_body = await response.json()
                    if json_body and json_body.get("resultset"):
                        respuestas_cda.append({"post_data": post_data, "json": json_body})
                except Exception:
                    pass

        page.on("response", interceptar_doquery)

        await page.goto(URL_DASHBOARD, wait_until="networkidle", timeout=30000)
        input_busqueda = page.locator("input[placeholder*='DNI' i], input[type='text']").first
        await input_busqueda.wait_for(state="visible", timeout=15000)
        await input_busqueda.fill(dni)
        await input_busqueda.press("Enter")

        # Esperar a que aparezca la tabla "Resultado de búsqueda"
        await page.wait_for_timeout(2500)
        await browser.close()

    # Buscar cuál de las respuestas interceptadas contiene los RUCs de la primera tabla
    for item in respuestas_cda:
        filas = item["json"].get("resultset", [])
        if filas and len(filas[0]) >= 2 and str(filas[0][0]).strip().isdigit() and len(str(filas[0][0]).strip()) == 11:
            print(f" -> [Interceptado OK] Parámetros reales de Pentaho: {item['post_data']}")
            for f in filas:
                proveedores.append({
                    "provider_ruc": str(f[0]).strip(),
                    "business_name": str(f[1]).strip(),
                    "condition": str(f[2]).strip() if len(f) >= 3 else "VIGENTE"
                })
            break

    return proveedores

async def obtener_detalle_por_ruc(client: httpx.AsyncClient, proveedor_base: Dict[str, Any]) -> Dict[str, Any]:
    ruc = proveedor_base["provider_ruc"]
    param_ruc = {"paramPARAM_RUC": ruc}

    res_vig, res_soc, res_adj = await asyncio.gather(
        pedir_cda(client, QUERIES_DETALLE["vigencias"], param_ruc),
        pedir_cda(client, QUERIES_DETALLE["socios"], param_ruc),
        pedir_cda(client, QUERIES_DETALLE["adjudicaciones"], param_ruc),
        return_exceptions=True
    )

    vigencias = [
        {
            "chapter": r.get("CAPITULORNP"),
            "status": r.get("ESTADO2"),
            "start_date": r.get("FINICIO"),
            "end_date": r.get("FFIN"),
            "department": r.get("DEPARTAMENTOOEE")
        }
        for r in matricial_a_dict(res_vig.get("raw") if isinstance(res_vig, dict) else None)
    ]

    socios = [
        {
            "relationship_type": r.get("TIPO"),
            "document_number": str(r.get("NRO_DOC", "")).strip(),
            "full_name": r.get("NOMBRE O RAZON SOCIAL"),
            "shares_percentage": float(r.get("ACCIONES", 0.0) or 0.0)
        }
        for r in matricial_a_dict(res_soc.get("raw") if isinstance(res_soc, dict) else None)
    ]

    adjudicaciones = [
        {
            "period": str(r.get("PERIODO")),
            "modality": r.get("MODALIDAD"),
            "entity": r.get("ENTIDAD"),
            "amount": float(r.get("MONTO", 0.0) or 0.0),
            "processes_count": int(r.get("CONVOCATORIAS", 0) or 0)
        }
        for r in matricial_a_dict(res_adj.get("raw") if isinstance(res_adj, dict) else None)
    ]

    return {
        "provider_ruc": ruc,
        "business_name": proveedor_base["business_name"],
        "condition": proveedor_base["condition"],
        "rnp_validities": vigencias,
        "shareholders": socios,
        "awards_history": adjudicaciones
    }

async def consultar_seace_arbol_por_dni(dni: str) -> Dict[str, Any]:
    dni_limpio = dni.strip()
    if len(dni_limpio) != 8 or not dni_limpio.isdigit():
        return {"error": "La consulta requiere exclusivamente un DNI numérico de 8 dígitos."}

    print(f"[1/3] Consultando presencia en el RNP para el DNI: {dni_limpio}...")
    inicio = datetime.now(timezone.utc)

    # Paso 1: Obtener la tabla maestra de proveedores vinculados al DNI
    proveedores_detectados = await descubrir_busqueda_dni_pentaho(dni_limpio)

    print(f"[2/3] Proveedores vinculados encontrados: {len(proveedores_detectados)}")
    for p in proveedores_detectados:
        print(f"   -> RUC: {p['provider_ruc']} | {p['business_name']} ({p['condition']})")

    # Paso 2: Para cada RUC detectado, extraer en paralelo sus 3 tablas hijas con httpx
    arbol_proveedores = []
    if proveedores_detectados:
        print(f"[3/3] Extrayendo árbol completo (Vigencias, Socios y Adjudicaciones) de {len(proveedores_detectados)} RUC(s)...")
        async with httpx.AsyncClient(headers=HEADERS_SEACE) as client:
            arbol_proveedores = await asyncio.gather(
                *(obtener_detalle_por_ruc(client, prov) for prov in proveedores_detectados)
            )

    duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)

    return {
        "audit": {
            "source": "SEACE_PENTAHO_CDA_DNI",
            "queried_dni": dni_limpio,
            "linked_providers_count": len(arbol_proveedores),
            "duration_ms": duracion,
            "status": "SUCCESS" if arbol_proveedores else "NO_DATA"
        },
        "data": {
            "dni": dni_limpio,
            "linked_providers": list(arbol_proveedores)
        }
    }

# Ejecución en el notebook
resultado_arbol = await consultar_seace_arbol_por_dni(DNI_PRUEBA)

print("\n================ ÁRBOL DE PROVEEDORES POR DNI ================")
for idx, prov in enumerate(resultado_arbol.get("data", {}).get("linked_providers", []), 1):
    print(f"\n[{idx}] PROVEEDOR: {prov['provider_ruc']} - {prov['business_name']} ({prov['condition']})")
    print(f"    ├── Vigencias RNP ({len(prov['rnp_validities'])}): {json.dumps(prov['rnp_validities'], ensure_ascii=False)}")
    print(f"    ├── Socios/Reps ({len(prov['shareholders'])}): {json.dumps(prov['shareholders'], ensure_ascii=False)}")
    print(f"    └── Adjudicaciones ({len(prov['awards_history'])} registros)")

[1/3] Consultando presencia en el RNP para el DNI: 44900952...
 -> [Interceptado OK] Parámetros reales de Pentaho: paramPARAM_DNIRUC0=44900952&path=%2Fpublic%2FANTECEDENTES_PROVEEDORES%2FANTECEDENTES_PROVEEDORES.cda&dataAccessId=QUERY_R_F0&outputIndexId=1&pageSize=0&pageStart=0&paramsearchBox=
[2/3] Proveedores vinculados encontrados: 2
   -> RUC: 20607441414 | INVERSION DEL SUR DEL PERU S.A.C. (VIGENTE)
   -> RUC: 20607524476 | INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA (VIGENTE)
[3/3] Extrayendo árbol completo (Vigencias, Socios y Adjudicaciones) de 2 RUC(s)...

================ ÁRBOL DE PROVEEDORES POR DNI ================

[1] PROVEEDOR: 20607441414 - INVERSION DEL SUR DEL PERU S.A.C. (VIGENTE)
    ├── Vigencias RNP (2): [{"chapter": "Proveedor de Servicios", "status": "VIGENTE", "start_date": "20/02/2021", "end_date": "INDETERMINADO", "department": "CUSCO"}, {"chapter": "Proveedor de Bienes", "status": "VIGENTE", "start_date": "20/02/202